# Domain E: Functional Connectivity and Circuit Interactions

This notebook addresses five questions:
- **E1:** Do noise correlations reveal cell-type-specific connectivity motifs?
- **E2:** Can transfer entropy / Granger causality reveal directed functional interactions?
- **E3:** How does population coupling relate to cell type and spatial position?
- **E4:** Does spontaneous activity structure during grey-screen epochs reveal cell-type-specific network dynamics? *(Zarr data)*
- **E5:** Do catch-trial responses reveal cell-type-specific expectation signals? *(Zarr data)*

**Data:** Zarr multimodal stores with ΔF/F traces (cells × trials), 3D coordinates, cell-type labels, gene expression, spontaneous activity, catch trials, and GLM data.

**Note:** E2 (Granger causality) requires continuous time-series data. The current dataset stores trial-level responses. Sections requiring continuous ΔF/F traces are marked `# ⚠️ REQUIRES: continuous time-series data`.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import kruskal, spearmanr, pearsonr, mannwhitneyu
from scipy.spatial.distance import cdist, pdist, squareform
import warnings
warnings.filterwarnings('ignore')

from functions.data_loading import load_mouse_zarr, load_zarr_10hz
from functions.analysis import xcorr_pair, pairwise_granger

sns.set_context('talk')
sns.set_style('whitegrid')

# ══════════════════════════════════════════════════════════════════════
# Load data from zarr multimodal stores
# ══════════════════════════════════════════════════════════════════════
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

adata_list = [load_mouse_zarr(mid, zarr_dir=ZARR_DIR, include_genes=True) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

coords = obs[['x_loc', 'y_loc', 'z_loc']].values.astype(float)
orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])

# Gene columns
META_COLS = {'unique_id', 'mouse_id', 'class_name', 'subclass_name', 'subclass_label',
             'supertype_name', 'cluster_name', 'cluster_alias', 'x_loc', 'y_loc', 'z_loc'}
GENE_COLS = [c for c in obs.columns if c not in META_COLS]

print(f"Total cells: {len(obs)}, Trials: {X.shape[1]}, Genes: {len(GENE_COLS)}")

## E4: Spontaneous Activity Assemblies (Grey-Screen Epochs)

Using the ~360 s spontaneous activity (GreyScreen) recorded at 10 Hz from zarr stores, detect neural assemblies via NMF, characterize their cell-type composition, and examine running-state modulation.

